# Synthetic AR(1) and Noise Time Series Dataset Demo

This notebook demonstrates the synthetic AR(1) time series dataset designed to evaluate and benchmark time series forecasting methods across diverse stochastic regimes. It incorporates rigorous AR(1) autoregressive processes with varying coefficients $\phi$, configurable noise levels, and evaluates 3-point moving average forecasting performance against a naive last-value baseline.

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'matplotlib==3.10.0', 'loguru==0.7.3')

In [ ]:
from loguru import logger
from pathlib import Path
import json
import sys
import matplotlib.pyplot as plt

logger.remove()
logger.add(sys.stdout, level="INFO", format="{time:HH:mm:ss}|{level:<7}|{message}")

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-29c492-empirical-audit-of-moving-average-baseli/main/round-2/dataset-1/demo/mini_demo_data.json"
import json, os

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded dataset group: {data['datasets'][0]['dataset']}")
print(f"Number of examples: {len(data['datasets'][0]['examples'])}")

## Configuration

Define configurable parameters for demonstration processing.

In [ ]:
# Tunable parameters
MAX_EXAMPLES = 10

## Processing and Standardizing Dataset Examples

Iterate through the dataset examples, parse input sequences, and extract forecasting metrics.

In [ ]:
logger.info("Starting data standardization and analysis for synthetic AR(1) dataset demo")

examples = []
raw_examples = data["datasets"][0]["examples"][:MAX_EXAMPLES]

for item in raw_examples:
    seq = json.loads(item["input"])
    output_val = float(item["output"])
    
    example = {
        "input_sequence": seq,
        "next_value": output_val,
        "phi": item["metadata_phi"],
        "sample_idx": item["metadata_sample_idx"],
        "mse_moving_average": item["metadata_mse_moving_average"],
        "mse_naive": item["metadata_mse_naive"],
        "improvement_over_naive": item["metadata_improvement_over_naive"]
    }
    examples.append(example)

logger.info(f"Processed {len(examples)} examples successfully.")

## Results and Visualization

Summarize and visualize the forecasting performance comparing 3-point moving average vs naive baseline across different $\phi$ coefficients.

In [ ]:
import pandas as pd

df = pd.DataFrame(examples)
print(df[["sample_idx", "phi", "mse_moving_average", "mse_naive", "improvement_over_naive"]].to_string())

# Plotting MSE comparison
plt.figure(figsize=(10, 5))
plt.plot(df.index, df["mse_moving_average"], marker='o', label='MSE Moving Average (3-pt)')
plt.plot(df.index, df["mse_naive"], marker='x', label='MSE Naive Baseline')
plt.xlabel('Sample Index')
plt.ylabel('Mean Squared Error (MSE)')
plt.title('Moving Average vs Naive Forecasting Error')
plt.legend()
plt.grid(True)
plt.show()